### Preparação das bases

In [ ]:
import pandas as pd
import pandas.io.sql as sqlio
import psycopg2 as ps
import os 

import warnings
warnings.filterwarnings('ignore')

In [2]:
conn = ps.connect(
        dbname='ANP',
        user=os.getenv('POSTGRES_USER'),
        password='postgres',
        host='localhost',
        port='5433'
    )


In [3]:
query = """
SELECT * FROM anp.preco_combustivel
"""

In [4]:
df = sqlio.read_sql_query(query, conn)

C:\Users\gabri\AppData\Local\Temp\ipykernel_15692\2233066866.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(query, conn)


In [5]:
df.head()

,regiao,estado,municipio,revenda,cnpj,nome_rua,numero_rua,complemento,bairro,cep,produto,data_coleta,valor_venda,unidade_medida,bandeira
0,NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. ...,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,None,CENTRO,62010-000,GASOLINA,2025-01-01,6.29,R$ / litro,RAIZEN
1,NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. ...,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,None,CENTRO,62010-000,GASOLINA ADITIVADA,2025-01-01,6.49,R$ / litro,RAIZEN
2,NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. ...,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,None,CENTRO,62010-000,DIESEL S10,2025-01-01,6.19,R$ / litro,RAIZEN
3,NE,CE,SOBRAL,ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. ...,08.775.979/0002-62,RUA TABELIÃO IDELFONSO CAVALCANTI,455,None,CENTRO,62010-000,ETANOL,2025-01-01,5.19,R$ / litro,RAIZEN
4,NE,CE,SOBRAL,V.C.EMPREENDIMENTOS LTDA,03.551.935/0002-35,AVENIDA JOSE EUCLIDES FERREIRA GOMES,30,POSTO FLASH,CORACAO DE JESUS,62043-070,GASOLINA,2025-01-01,6.53,R$ / litro,RAIZEN


In [6]:
df.shape

(4657284, 15)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4657284 entries, 0 to 4657283
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   regiao          object 
 1   estado          object 
 2   municipio       object 
 3   revenda         object 
 4   cnpj            object 
 5   nome_rua        object 
 6   numero_rua      object 
 7   complemento     object 
 8   bairro          object 
 9   cep             object 
 10  produto         object 
 11  data_coleta     object 
 12  valor_venda     float64
 13  unidade_medida  object 
 14  bandeira        object 
dtypes: float64(1), object(14)
memory usage: 533.0+ MB


In [ ]:
# transforma a coluna direto no df para datetime
df['data_coleta'] = pd.to_datetime(df['data_coleta'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4657284 entries, 0 to 4657283
Data columns (total 15 columns):
 #   Column          Dtype         
---  ------          -----         
 0   regiao          object        
 1   estado          object        
 2   municipio       object        
 3   revenda         object        
 4   cnpj            object        
 5   nome_rua        object        
 6   numero_rua      object        
 7   complemento     object        
 8   bairro          object        
 9   cep             object        
 10  produto         object        
 11  data_coleta     datetime64[ns]
 12  valor_venda     float64       
 13  unidade_medida  object        
 14  bandeira        object        
dtypes: datetime64[ns](1), float64(1), object(13)
memory usage: 533.0+ MB


In [11]:
# verifica se o df possui dados nulos e agrupa por colunas
df.isnull().sum()

regiao                  0
estado                  0
municipio               0
revenda                 0
cnpj                    0
nome_rua                0
numero_rua           1727
complemento       3609498
bairro              10654
cep                     0
produto                 0
data_coleta       4236875
valor_venda       4236875
unidade_medida          0
bandeira                0
dtype: int64

In [28]:
# seleção de colunas especificas
df_anp = df[['data_coleta', 'regiao', 'estado', 'municipio', 'bandeira', 'produto', 'valor_venda']]

In [29]:
df_anp.head()

,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda
0,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA,6.29
1,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA ADITIVADA,6.49
2,2025-01-01,NE,CE,SOBRAL,RAIZEN,DIESEL S10,6.19
3,2025-01-01,NE,CE,SOBRAL,RAIZEN,ETANOL,5.19
4,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA,6.53


In [30]:
# adiciona colunas de ano e mes baseando-se na coluna data_coleta, usando a função dt (datetime) para classificar os valores corretamente
df_anp['ano'] = df_anp['data_coleta'].dt.year
df_anp['mes'] = df_anp['data_coleta'].dt.month

In [31]:
df_anp.head()

,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda,ano,mes
0,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA,6.29,2025.0,1.0
1,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA ADITIVADA,6.49,2025.0,1.0
2,2025-01-01,NE,CE,SOBRAL,RAIZEN,DIESEL S10,6.19,2025.0,1.0
3,2025-01-01,NE,CE,SOBRAL,RAIZEN,ETANOL,5.19,2025.0,1.0
4,2025-01-01,NE,CE,SOBRAL,RAIZEN,GASOLINA,6.53,2025.0,1.0


In [32]:
df_anp.describe().round(2)

,data_coleta,valor_venda,ano,mes
count,420409,420409.00,420409.0,420409.00
mean,2025-04-02 06:53:17.337830400,5.89,2025.0,3.54
min,2025-01-01 00:00:00,3.19,2025.0,1.00
25%,2025-02-17 00:00:00,5.59,2025.0,2.00
50%,2025-04-01 00:00:00,6.17,2025.0,4.00
75%,2025-05-19 00:00:00,6.49,2025.0,5.00
max,2025-06-30 00:00:00,9.49,2025.0,6.00
std,NaN,0.89,0.0,1.70


### Inicio do Storytelling

Tipos de combustíveis que são comercializados

In [33]:
# visualizando valores únicos
print(f'Os produtos comercializados são: {df_anp.produto.unique()}')

Os produtos comercializados são: ['GASOLINA' 'GASOLINA ADITIVADA' 'DIESEL S10' 'ETANOL' 'DIESEL' 'GNV']


Anos que estão sendo analisados

In [35]:
print(f'Os anos da nossa base: {df_anp.ano.unique()}')

Os anos da nossa base: [2025.   nan]


Valores minimos, máximos e médios dos produtos por ano

In [ ]:
df_anp_valor = df_anp[['ano', 'produto', 'valor_venda']]

In [ ]:
df_anp_valor.groupby(['produto', 'ano']).agg(['min', 'max', 'mean']).round(2)

In [ ]:
df_anp_valor_estado = df_anp[['ano', 'produto', 'estado', 'valor_venda']]

In [ ]:
df_anp_valor.groupby(['produto', 'ano', 'estado']).agg(['min', 'max', 'mean']).round(2)